# Invoice PDF → Result
Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Upload one PDF when prompted. The final invoice JSON is displayed and downloaded automatically. Missing or uncertain fields remain marked for review.


In [ ]:
#@title 1. Load project
import os
import pathlib
import subprocess

PROJECT_REF = 'main' #@param {type:"string"}
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', PROJECT_REF, '--depth', '1', 'https://github.com/ubaid-148/OCR.git', str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / '.git').is_dir():
    current_branch = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True).strip()
    if current_branch != PROJECT_REF:
        raise RuntimeError(f'Existing clone is on {current_branch}, expected {PROJECT_REF}. Start a fresh runtime to test the selected branch.')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)


In [ ]:
#@title 2. Prepare OCR
import os, pathlib, shutil, subprocess, sys
REQUIRE_GPU = True #@param {type:"boolean"}
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('GPU attached:', gpu_runtime, flush=True)
if REQUIRE_GPU and not gpu_runtime:
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run all cells again.')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
os.environ['OCR_TARGETED_RETRY'] = 'false'
os.environ['OCR_FORCE_RASTER'] = 'true'
os.environ['USE_LOCAL_AI'] = 'false'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Ready. Upload your invoice in the next cell.')


In [ ]:
#@title 3. Upload PDF and get invoice result
import json
import os
import re
import subprocess
import tempfile
from pathlib import Path
from google.colab import files

OCR_LANGUAGES = 'eng+ara' #@param ['eng+ara', 'eng', 'ara', 'eng+urd']
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run the setup cells first.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PDF.')
original_name, pdf_bytes = next(iter(uploaded.items()))
if Path(original_name).suffix.lower() != '.pdf' or not pdf_bytes.startswith(b'%PDF-'):
    raise ValueError('Please upload a valid PDF.')

with tempfile.TemporaryDirectory(prefix='invoice-result-') as temporary:
    run_dir = Path(temporary)
    pdf_path = run_dir / 'input.pdf'
    raw_path = run_dir / 'raw_ocr.json'
    clean_path = run_dir / 'invoice.json'
    pdf_path.write_bytes(pdf_bytes)
    commands = [
        [OCR_PYTHON, '-u', str(PROJECT_DIR / 'coordinate_ocr.py'),
         str(pdf_path), str(raw_path), OCR_LANGUAGES],
        [OCR_PYTHON, str(PROJECT_DIR / 'main.py'), str(raw_path),
         '--output', str(clean_path), '--no-llm'],
    ]
    print('Reading invoice…')
    for command in commands:
        process = subprocess.run(command, cwd=PROJECT_DIR, env=os.environ.copy(),
                                 capture_output=True, text=True)
        if process.returncode:
            raise RuntimeError(process.stderr[-12000:] or process.stdout[-12000:] or 'Invoice extraction failed')
    invoice = json.loads(clean_path.read_text(encoding='utf-8'))

results_dir = PROJECT_DIR / 'benchmark_outputs' / 'invoices'
results_dir.mkdir(parents=True, exist_ok=True)
filename = re.sub(r'[^A-Za-z0-9_.-]', '_', Path(original_name).stem) + '-invoice.json'
invoice_path = results_dir / filename
invoice_path.write_text(json.dumps(invoice, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(invoice, ensure_ascii=False, indent=2))
files.download(str(invoice_path))
